# Task 1: Preparing the data

In this task you'll take the raw power-flow data and turn it into a clean, hourly time series that a forecasting model can use. Task 2 starts from the file you save at the end.

**By the end you should have:**

- a file `data/forecast_preproc_task.csv` with three columns (`Timestamp`, `Demand`, `Import`) and one row per hour;
- a short explanation, in your own words, of each step you took and why.
- You can compare your .csv to `data/forecast_preproc.csv` to confirm your solution is ready.

**How this notebook works**

- Read the text, then run each cell in order (Shift + Enter).
- Cells starting with **`# TODO`** need you to write a little code.
- **Check** cells test your work. If one fails, read the message, fix the cell above it and run both again.
- **Question** cells ask for a short written answer. Double-click the *Your answer* cell below each one to edit it.

Run the notebook from the project folder (the one that contains `data/`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Step 1: Load the data

The raw data is in `data/Forecasting_dataset.csv`. It has a reading every 5 minutes for six quantities: `Demand`, `Generation`, `Import`, `Solar`, `Wind` and `Other`. We only forecast **Demand** and **Import**.

**Watch the dates.** The timestamps are written day first, so `01/12/2020` means 1 December. We tell pandas this with `dayfirst=True`. Without it, pandas may silently read the date as 12 January.

In [ ]:
data = pd.read_csv('data/Forecasting_dataset.csv')
data['Timestamp'] = pd.to_datetime(data['Timestamp'], dayfirst=True)
data = data.set_index('Timestamp')
data.head()

Print how many rows there are, and the first and last timestamp.

In [ ]:
# TODO: print the number of rows, and the first and last timestamp

**Question 1.** Roughly how many weeks of data is this? At one reading every 5 minutes, about how many rows would you expect? Keep this in mind for step 3.

*Your answer:*

## Step 2: Look before you clean

Always plot the data before changing it. Every cleaning step later in this notebook should be justified by something you can see here.

Plot `Demand` and `Import` against time on the same axes. Give each a `label`, add a title and axis labels, and call `ax.legend()`.

In [ ]:
# TODO: plot Demand and Import against time on the same axes (with labels, title and a legend)
fig, ax = plt.subplots(figsize=(13, 4))

**Question 2.** Describe what you see.

- Is there a daily pattern? A weekly one?
- Does anything look physically implausible?

*Your answer:*

## Step 3: Find the missing data

Missing data comes in two kinds, and a plot like the one above hides both:

1. **Blank values.** The timestamp is in the file but the value is empty. pandas reads these as `NaN`. matplotlib leaves a break in the line, but a short break is far too narrow to see over three and a half months. (You can also see these easily by opening the .csv file directly.)
2. **Missing rows.** The timestamp isn't in the file at all. matplotlib draws a straight line across the gap, so it's invisible too.

Start with blank values. Count how many there are in `Demand` and `Import`.

In [ ]:
# TODO: count the blank values in the Demand and Import columns

To find missing rows, compare the data with a perfect 5-minute grid running from the first timestamp to the last. `pd.date_range` builds that grid for us.

In [ ]:
expected_index = pd.date_range(start=data.index.min(), end=data.index.max(), freq='5min')

# TODO: how many rows are missing? Compare len(expected_index) with len(data).
n_missing_rows = None # TODO fill me in with the correct expression
print('Missing rows:', n_missing_rows)

In [ ]:
# Check (don't edit me)
assert n_missing_rows is not None, 'Set n_missing_rows in the cell above.'
assert n_missing_rows == 288, f'Got {n_missing_rows}. Check which length you subtract from which.'
print('Looks good.')

Now find *where* the missing data is.

- For the **missing rows**, the gap between one timestamp and the next is more than 5 minutes. `data.index.to_series().diff()` gives the gap before each row; find the rows where it's bigger than `pd.Timedelta('5min')`.
- For the **blank values**, count the `NaN`s per calendar day. Group `data['Demand'].isna()` by `data.index.date` and sum. Show only the days that have some.

In [ ]:
# TODO: print the missing-row gaps (where the step between timestamps is > 5 min)


# TODO: print the number of blank Demand values per day, for days that have any

Zoom in on the week with the most missing data. Each dot is one reading. Plot `Demand` for 14 to 20 February 2021 with `marker='.'` so individual readings show up.

In [ ]:
# TODO: plot Demand for the week of 14-20 Feb 2021, with dots marking each reading
window = data.loc['2021-02-14':'2021-02-20']

**Question 3.**

- Where is the longest stretch of missing data, and how long is it?
- In the zoomed plot, how does matplotlib draw the missing rows? How does it draw the blank values? Why is the first one misleading?

*Your answer:*

## Step 4: Put the data on a regular grid

Time-series models assume one row for every time step, evenly spaced. With rows missing, the model can't tell that a day has passed between two neighbouring rows.

The fix is to `reindex` the data onto the `expected_index` grid from step 3. Timestamps that were missing come back as rows full of `NaN`. The gap is now explicit instead of hidden.

In [ ]:
# TODO: reindex data onto expected_index, then set the index name back to 'Timestamp'


print('Rows now:', len(data))
print('NaNs per column:')
print(data[['Demand', 'Import']].isna().sum())

In [ ]:
# Check (don't edit me)
assert len(data) == 31968, f'Expected 31968 rows on the full grid, got {len(data)}.'
assert pd.infer_freq(data.index) == '5min', 'Timestamps should now be exactly 5 minutes apart.'
assert (data[['Demand', 'Import']].isna().sum() == 383).all(), 'Each column should have 383 NaNs (288 missing rows + 95 blanks).'
print('Looks good.')

**Question 4.** How many `NaN`s does each column have now? Explain where that number comes from, using your answers from step 3.

*Your answer:*

## Step 5: Remove the fault

In step 2 you may have spotted a reading that drops to a fraction of its normal level. It's at **2021-02-15 00:40**, five minutes before the 24-hour outage starts, so it's almost certainly the same equipment fault. A model trained on it would try to explain it, which would distort the forecast.

We'll use a simple rule: a value is an **outlier** if it is at least **4 standard deviations** from the column's mean:

$$|x - \mu| \ge 4\sigma$$

We set outliers to `NaN` rather than deleting the row. That keeps the grid regular, and the value is treated as missing, like the gaps.

In [ ]:
# TODO: for each of 'Demand' and 'Import':
#   1. work out the column's mean and standard deviation (.mean() and .std())
#   2. build a True/False mask of values at least 4 standard deviations from the mean
#      (.abs() gives absolute values)
#   3. set those values to NaN:  data.loc[mask, column] = np.nan
# Print how many values you removed from each column.

In [ ]:
# Check
fault = pd.Timestamp('2021-02-15 00:40')
assert data.loc[fault, ['Demand', 'Import']].isna().all(), 'The fault reading at 2021-02-15 00:40 is still there.'
assert (data[['Demand', 'Import']].isna().sum() == 384).all(), (
    'Each column should have exactly one more NaN than in step 4. Did you remove too much, or too little?')
print('Looks good.')

**Question 5.**

- The tallest peaks in the raw plot (around 12,850) were *not* removed. Why is that the right outcome?
- The mean and standard deviation were calculated from all the data, *including* the fault. Why could this rule struggle on a dataset with many faults, or with a fault that lasts for hours?

*Your answer:*

## Step 6: Experiment. Should we fill the gaps?

> **This step is an experiment.** You'll try a common technique, judge the result, and then decide whether to keep it.

**Interpolation** fills each `NaN` using the values around it. *Linear* interpolation draws a straight line from the last known value before a gap to the first known value after it. Many models (most of scikit-learn, neural networks) can't accept `NaN`s, so this is a very common step.

Try it on a **copy** of the data, so the main `data` isn't changed.

In [ ]:
# TODO: make a copy of data called data_interp, and fill its NaNs with linear interpolation (pandas has a tool)
data_interp = data.copy()
data_interp = None # Fill me in

Compare the two versions on a long gap and on a short one. Orange shows where interpolation filled in values.

In [ ]:
windows = [('2021-02-14', '2021-02-17', 'Long gap: the 24-hour outage'),
           ('2021-02-19 01:00', '2021-02-19 10:00', 'Short gap: 3 hours of blank values')]
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, (start, end, title) in zip(axes, windows):
    ax.plot(data_interp.loc[start:end, 'Demand'], color='tab:orange', label='interpolated')
    ax.plot(data.loc[start:end, 'Demand'], color='tab:blue', label='original (gaps left as NaN)')
    ax.set_title(title)
    ax.set_ylabel('Demand')
    ax.tick_params(axis='x', rotation=30)
    ax.legend()
plt.tight_layout()
plt.show()

**Question 6.**

1. Does interpolation make the data look **better or worse**? Answer separately for the short gap and the long gap.
2. What are the **pros** of filling gaps this way?
3. What are the **cons**? Think about what a model would learn from a straight line drawn across a full day of demand.
4. Is there a gap length beyond which you'd stop trusting interpolation? Roughly what length, and why?

*Your answer:*

### Our decision for this task: don't interpolate

The model we use in Task 2, **SARIMAX**, doesn't need gap-free data. Roughly speaking, it steps through the series one time step at a time, updating its estimate of what's going on at each step. When it meets a `NaN`, it skips the update and carries its own prediction forward. So we can leave the gaps as they are, rather than inventing a day of data.

**From here on, carry on with `data`, not `data_interp`.** If you later use a model that can't handle `NaN`s, this is the step to revisit.

## Step 7: Downsample to hourly

At 5-minute resolution there are about 32,000 rows. For a week-ahead forecast that's far more detail than we need, and it makes the model slow to fit. Hourly data keeps the daily and weekly shape and is 12 times smaller.

There are two reasonable ways to do this:

- **Option A:** keep the reading taken on the hour, with `data.iloc[::N]` (every Nth row). How many 5-minute readings are there in an hour?
- **Option B:** average each hour, with `data.resample('h').mean()`.

Either is fine. Say which you chose, and why, in your summary at the end.

In [ ]:
# TODO: make an hourly version of data called data_hourly
data_hourly = None
print('Rows after downsampling:', None if data_hourly is None else len(data_hourly))

In [ ]:
# Check
assert data_hourly is not None, 'Set data_hourly in the cell above.'
assert len(data_hourly) == 2664, f'Expected one row per hour (2664 rows), got {len(data_hourly)}.'
assert pd.infer_freq(data_hourly.index) in ('h', 'H'), 'Timestamps should be exactly one hour apart.'
print('Looks good.')

**Question 7.** The fault reading was at 00:40.

- With option A, would the fault have made it into the hourly data anyway? With option B?
- Why is it still good practice to remove outliers *before* downsampling?

*Your answer:*

## Step 8: Save the result

Keep only the `Demand` and `Import` columns of `data_hourly` and save them as a CSV. Task 2 reads this file.

Remember: save the version with the gaps left as `NaN`, not the interpolated one.

In [ ]:
OUTPUT_PATH = 'data/forecast_preproc_task.csv'
# TODO: keep only the 'Demand' and 'Import' columns of data_hourly, and save them to OUTPUT_PATH with .to_csv()

In [ ]:
# Check
saved = pd.read_csv(OUTPUT_PATH, index_col='Timestamp', parse_dates=True)
assert list(saved.columns) == ['Demand', 'Import'], f'Expected columns Demand and Import, got {list(saved.columns)}.'
assert len(saved) == 2664, f'Expected 2664 hourly rows, got {len(saved)}.'
assert pd.infer_freq(saved.index) in ('h', 'H'), 'Timestamps should be one hour apart.'
assert saved.isna().sum().sum() > 0, (
    'There are no gaps at all. Did you save the interpolated data? SARIMAX handles NaNs, so keep them.')
print('Saved and checked. You are ready for Task 2.')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(saved.index, saved['Demand'], label='Demand')
ax.plot(saved.index, saved['Import'], label='Import')
ax.set_title('Cleaned hourly data')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Power')
ax.legend()
plt.show()

## Step 9: Explain what you did

Compare the cleaned plot with the raw one from step 2. Then fill in the table below, in your own words. Aim for one or two sentences per cell: someone who hasn't seen this notebook should understand what you did and why.

| Step | What I did | Why |
|---|---|---|
| Load the data | | |
| Look at the raw data | | |
| Find the missing data | | |
| Regular grid | | |
| Remove the fault | | |
| Interpolation (tried) | | |
| Downsample | | |
| Save | | |

**Question 8.** Beyond the data you were given, what other datasets or outside factors might help forecast demand and import? Think about what makes people use more or less electricity, and what affects how much is generated locally.

*Your answer:*